# 🎚️ Routing by Model Tier

The routing pattern usually sends a request to a specialised *handler* — story, poem, joke (`01_routing_langgraph.ipynb`). This notebook routes on a different axis: it sends the request to a specialised **model tier**. A cheap model looks at each request and decides whether it is hard. Easy work goes to the cheap model; hard work goes to a reasoning model. You pay reasoning prices only on the traffic that needs them.

## Learning Objectives
In this notebook, you will learn:
1. **Two model tiers** - Configure a cheap model and a reasoning model through the same factory
2. **A tier router** - Build a LangGraph workflow whose conditional edge selects a model, not a handler
3. **Measuring the router** - Compare routed accuracy and reasoning-call count against all-cheap and all-reasoning baselines
4. **The router's own errors** - Read the router's confusion matrix, and choose a fallback direction deliberately

## Where this fits
Same graph shape as `01_routing_langgraph.ipynb`, different axis. The *which model* and *how much effort* trade-offs behind it are measured on fixed tasks in the *Reasoning and Model Selection* notebooks of [repository 01 — foundations](https://github.com/Sourav692/ai-engineering-demystified-01-foundations); this notebook makes the choice **per request, at runtime**, and measures whether the routing actually paid.

## Prerequisites
- The routing pattern — `01_routing_langgraph.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env` (no other keys needed)
- Cost: roughly `3 × len(WORKLOAD)` answering calls plus triage — about 30 calls, a few cents

---
## 📦 Step 1: Environment Setup

Load the API key, then create **two** chat models through the course factory. This is the one notebook in the module that deliberately uses a second model: the tier *is* the lesson.

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

✅ Environment ready


Two things about the reasoning tier will bite you on the first real call, not at construction time:

1. **o-series models reject a custom temperature.** `get_openai_llm` defaults to `temperature=0`, which is right for the cheap tier and an error for the reasoning tier, so the reasoning tier passes `temperature=1`.
2. **`reasoning_effort` only means something to a reasoning model.** The factory forwards extra keyword arguments to `ChatOpenAI`, so `reasoning_effort="high"` reaches `o4-mini`. The two tiers use the same provider and the same prompts, so any measured difference comes from the model alone.

In [2]:
# ============================================================================
# LLM INITIALIZATION: A cheap tier and a reasoning tier
# ============================================================================
from aied_helpers import get_openai_llm

# Cheap tier: answers easy requests directly, and also does the triage.
cheap_llm = get_openai_llm(model="gpt-4o-mini")

# Reasoning tier: temperature=1 because o-series rejects anything else;
# reasoning_effort is passed through the factory to ChatOpenAI.
reasoning_llm = get_openai_llm(model="o4-mini", temperature=1, reasoning_effort="high")

print(f"✅ Cheap tier:     {cheap_llm.model_name}")
print(f"✅ Reasoning tier: {reasoning_llm.model_name} (reasoning_effort={reasoning_llm.reasoning_effort})")

✅ Cheap tier:     gpt-4o-mini
✅ Reasoning tier: o4-mini (reasoning_effort=high)


---
## 🏷️ Part 1: The Triage Schema and State

The router has exactly one job: decide `easy` or `hard`. Keeping the decision to a two-value enum matters — a router that can emit free text is a router that can emit something your graph has no edge for.

In [3]:
# ============================================================================
# TRIAGE SCHEMA: The router's only output is easy | hard
# ============================================================================
from typing import Literal

from pydantic import BaseModel, Field


class Triage(BaseModel):
    """How hard is this request?"""

    difficulty: Literal["easy", "hard"] = Field(
        description=(
            "'hard' if answering needs multi-step reasoning, constraint satisfaction, "
            "or arithmetic across several dependent steps. "
            "'easy' if it is lookup, extraction, formatting, or a single obvious step."
        )
    )


triage_router = cheap_llm.with_structured_output(Triage)

print("✅ Triage router ready")

✅ Triage router ready


In [4]:
# ============================================================================
# GRAPH STATE: What each request carries through the router
# ============================================================================
from typing_extensions import TypedDict


class TierState(TypedDict):
    question: str        # the incoming request
    difficulty: str      # what triage decided
    answer: str          # final answer
    tier_used: str       # which model actually answered — for the cost accounting
    seconds: float       # wall-clock for the answering call only


print(f"✅ TierState fields: {list(TierState.__annotations__)}")

✅ TierState fields: ['question', 'difficulty', 'answer', 'tier_used', 'seconds']


---
## 🧭 Part 2: The Tier Router

Three nodes: one triages, two answer. The answering nodes are deliberately identical except
for the model they call — that is the whole experiment. If they differed in prompt as well,
any measured difference would be unattributable.

In [5]:
# ============================================================================
# NODES: triage, two answering tiers, and the conditional edge
# ============================================================================
import time

from langchain_core.messages import HumanMessage, SystemMessage


def triage(state: TierState) -> dict:
    """Cheap model decides whether this needs the expensive tier."""
    decision = triage_router.invoke([
        SystemMessage(content="Classify how hard this request is to answer correctly."),
        HumanMessage(content=state["question"]),
    ])
    return {"difficulty": decision.difficulty}


def _answer_with(llm, state: TierState, tier: str) -> dict:
    started = time.perf_counter()
    reply = llm.invoke([HumanMessage(content=state["question"])])
    return {
        "answer": (reply.content or "").strip(),
        "tier_used": tier,
        "seconds": time.perf_counter() - started,
    }


def answer_cheap(state: TierState) -> dict:
    return _answer_with(cheap_llm, state, "cheap")


def answer_reasoning(state: TierState) -> dict:
    return _answer_with(reasoning_llm, state, "reasoning")


def pick_tier(state: TierState) -> Literal["answer_cheap", "answer_reasoning"]:
    """The conditional edge. Selects a model tier, not a handler."""
    return "answer_reasoning" if state["difficulty"] == "hard" else "answer_cheap"


print("✅ Nodes and tier selector defined")

✅ Nodes and tier selector defined


In [6]:
# ============================================================================
# BUILD THE GRAPH: triage → (cheap | reasoning) → END
# ============================================================================
from langgraph.graph import END, START, StateGraph

builder = StateGraph(TierState)
builder.add_node("triage", triage)
builder.add_node("answer_cheap", answer_cheap)
builder.add_node("answer_reasoning", answer_reasoning)

builder.add_edge(START, "triage")
builder.add_conditional_edges("triage", pick_tier)
builder.add_edge("answer_cheap", END)
builder.add_edge("answer_reasoning", END)

router_graph = builder.compile()
print("✅ Tier router compiled")

✅ Tier router compiled


In [7]:
# ============================================================================
# INSPECT THE GRAPH: Print nodes and edges (no network call needed)
# ============================================================================
g = router_graph.get_graph()

print("🧩 Nodes:", [n for n in g.nodes if not n.startswith("__")])
print("🔗 Edges:")
for e in g.edges:
    kind = "conditional" if e.conditional else "fixed"
    print(f"   {e.source:>18} ──▶ {e.target:<18} ({kind})")

🧩 Nodes: ['triage', 'answer_cheap', 'answer_reasoning']
🔗 Edges:
            __start__ ──▶ triage             (fixed)
               triage ──▶ answer_cheap       (conditional)
               triage ──▶ answer_reasoning   (conditional)
         answer_cheap ──▶ __end__            (fixed)
     answer_reasoning ──▶ __end__            (fixed)


Because `pick_tier` is annotated `Literal["answer_cheap", "answer_reasoning"]`, LangGraph can infer the conditional edges without a mapping dict — compare the explicit map in `01_routing_langgraph.ipynb`.

---
## 📊 Part 3: Measuring the Router Against Both Baselines

The workload has to be **mixed**, and honestly mixed. If it is all hard, routing cannot help
and you will conclude the pattern is worthless. If it is all easy, routing looks free and
you will conclude it is magic. Real traffic is lopsided — mostly easy, occasionally not —
so this set is too.

Scoring is a **token match**, not a substring match: the answer is split into tokens and the expected value must be one of them. A substring check would count `"c"` as found in almost any sentence.

In [8]:
# ============================================================================
# WORKLOAD: mostly easy, occasionally hard — with ground truth
# ============================================================================
# (question, expected-substring, is_actually_hard) — the third field is ground truth for
# scoring the router itself, not something the router sees.
WORKLOAD: list[tuple[str, str, bool]] = [
    ("Extract the order id from: 'Order #A-4417 shipped Tuesday.' Reply with the id only.",
     "A-4417", False),
    ("What is the capital of Portugal? One word.", "lisbon", False),
    ("Convert 2:45 pm to 24-hour time. HH:MM only.", "14:45", False),
    ("Reply with the word 'ok' and nothing else.", "ok", False),
    ("Services deploy one per day Mon-Fri. auth before billing. search the day immediately "
     "after billing. notify Monday. reporting not Friday. Which deploys Friday? One word.",
     "search", True),
    # After A B C D, re-accessing A makes B the least recently used — so E evicts B.
    ("A cache holds 4 entries, LRU eviction. Accesses: A B C D A E B. Which key is evicted "
     "when E is inserted? One letter.", "b", True),
    ("Retries use exponential backoff 2s, 4s, 8s. Total seconds waiting across all three? "
     "Number only.", "14", True),
]

print(f"{len(WORKLOAD)} requests — "
      f"{sum(1 for *_, h in WORKLOAD if not h)} easy, {sum(1 for *_, h in WORKLOAD if h)} hard")

7 requests — 4 easy, 3 hard


### 3.1 🏃 Run the router, then both baselines

Three passes over the same workload. Only by measuring all three can you say whether routing
paid — "the router was cheap" means nothing without "cheaper than what".

In [9]:
# ============================================================================
# THREE PASSES: routed, all-cheap, all-reasoning
# ============================================================================
import re


def is_correct(expected: str, answer: str) -> bool:
    """Token match: the expected value must appear as a whole token of the answer."""
    tokens = re.findall(r"[a-z0-9:\-]+", answer.lower())
    return expected.lower() in tokens


def run_routed() -> list[dict]:
    out = []
    for question, expected, truly_hard in WORKLOAD:
        state = router_graph.invoke({"question": question})
        out.append({
            "expected": expected, "truly_hard": truly_hard,
            "tier": state["tier_used"], "routed_hard": state["difficulty"] == "hard",
            "correct": is_correct(expected, state["answer"]),
            "seconds": state["seconds"],
        })
    return out


def run_fixed(llm, tier: str) -> list[dict]:
    out = []
    for question, expected, truly_hard in WORKLOAD:
        started = time.perf_counter()
        reply = llm.invoke([HumanMessage(content=question)])
        text = (reply.content or "").strip()
        out.append({
            "expected": expected, "truly_hard": truly_hard,
            "tier": tier, "routed_hard": tier == "reasoning",
            "correct": is_correct(expected, text),
            "seconds": time.perf_counter() - started,
        })
    return out


routed = run_routed()
all_cheap = run_fixed(cheap_llm, "cheap")
all_reasoning = run_fixed(reasoning_llm, "reasoning")
print("✅ Three passes complete")

✅ Three passes complete


In [10]:
# ============================================================================
# THE PAYOFF TABLE: accuracy vs reasoning calls per strategy
# ============================================================================
def report(name: str, rows: list[dict]) -> None:
    acc = sum(r["correct"] for r in rows) / len(rows)
    secs = sum(r["seconds"] for r in rows)
    calls = f'{sum(1 for r in rows if r["tier"] == "reasoning")}/{len(rows)}'
    print(f"{name:<16}{acc:>9.0%}{secs:>11.1f}{calls:>18}")


print(f"{'strategy':<16}{'accuracy':>9}{'total sec':>11}{'reasoning calls':>18}")
print("-" * 54)
report("all cheap", all_cheap)
report("routed", routed)
report("all reasoning", all_reasoning)

print("\nRouting is worthwhile when its accuracy is at or near all-reasoning,")
print("while its reasoning-call count is far closer to all-cheap.")

strategy         accuracy  total sec   reasoning calls
------------------------------------------------------
all cheap             71%        5.2               0/7
routed                86%       13.4               1/7
all reasoning        100%       20.1               7/7

Routing is worthwhile when its accuracy is at or near all-reasoning,
while its reasoning-call count is far closer to all-cheap.


The shape you are looking for: **routed accuracy tracks all-reasoning, routed reasoning-call
count tracks all-cheap.** That gap is the saving, and it is the entire argument for the
pattern.

If routed accuracy sits below all-reasoning, the router is misclassifying hard questions as
easy — see the next section. If routed cost sits near all-reasoning, the router is calling
everything hard, and you are paying for triage on top of reasoning: strictly worse than
having no router.

---
## ⚠️ Part 4: The Router Is a Model Call, and It Can Be Wrong

This is the part that separates a demo from a design. The triage step does not observe
difficulty, it *predicts* it — so it has a confusion matrix, and the two mistakes cost
wildly different amounts.

In [11]:
# ============================================================================
# ROUTER CONFUSION MATRIX: where did triage send each request?
# ============================================================================
tp = sum(1 for r in routed if r["truly_hard"] and r["routed_hard"])
fn = sum(1 for r in routed if r["truly_hard"] and not r["routed_hard"])
fp = sum(1 for r in routed if not r["truly_hard"] and r["routed_hard"])
tn = sum(1 for r in routed if not r["truly_hard"] and not r["routed_hard"])

print(f"  hard, sent to reasoning (right)      : {tp}")
print(f"  easy, sent to cheap     (right)      : {tn}")
print(f"  easy, sent to reasoning (overspend)  : {fp}   <- costs money")
print(f"  hard, sent to cheap     (wrong answer): {fn}   <- costs correctness")

missed = [r for r in routed if r["truly_hard"] and not r["routed_hard"]]
if missed:
    print(f"\n{len(missed)} hard question(s) were routed cheap; "
          f"{sum(1 for r in missed if not r['correct'])} of those came back wrong.")

print("\nPer-request decisions:")
for (question, *_), r in zip(WORKLOAD, routed):
    mark = "✅" if r["correct"] else "❌"
    print(f"  {mark} truly {'hard' if r['truly_hard'] else 'easy'} → {r['tier']:<9} | {question[:60]}")

  hard, sent to reasoning (right)      : 1
  easy, sent to cheap     (right)      : 4
  easy, sent to reasoning (overspend)  : 0   <- costs money
  hard, sent to cheap     (wrong answer): 2   <- costs correctness

2 hard question(s) were routed cheap; 1 of those came back wrong.

Per-request decisions:
  ✅ truly easy → cheap     | Extract the order id from: 'Order #A-4417 shipped Tuesday.' 
  ✅ truly easy → cheap     | What is the capital of Portugal? One word.
  ✅ truly easy → cheap     | Convert 2:45 pm to 24-hour time. HH:MM only.
  ✅ truly easy → cheap     | Reply with the word 'ok' and nothing else.
  ✅ truly hard → reasoning | Services deploy one per day Mon-Fri. auth before billing. se
  ❌ truly hard → cheap     | A cache holds 4 entries, LRU eviction. Accesses: A B C D A E
  ✅ truly hard → cheap     | Retries use exponential backoff 2s, 4s, 8s. Total seconds wa


### 4.1 🧮 The two errors are not symmetric

- **Easy sent to reasoning** wastes money. Annoying, bounded, and visible on a bill.
- **Hard sent to cheap** returns a confidently wrong answer. Unbounded, and invisible until
  someone downstream acts on it.

So the router should be **biased toward over-escalating**. Tune the triage prompt so
borderline cases land in `hard`, and accept the overspend as insurance.

That asymmetry also fixes the fallback direction. The tempting design is *"try cheap, detect
a bad answer, retry on reasoning"* — but detecting a confidently wrong answer is exactly the
hard problem you do not have a solution for. **The workable direction is to escalate on
uncertainty before answering, not to demote after.** If you can reliably tell that an answer
is wrong, you did not need the router; you needed that detector.

---
## 📝 Summary

In this notebook, we learned:

### 1. Two model tiers
- The same factory builds both tiers; the reasoning tier needs `temperature=1` and takes `reasoning_effort`
- Keep prompts identical across tiers so a measured difference is attributable to the model

### 2. A tier router
- Routing by model tier is the routing pattern on a different axis: the conditional edge picks a model, not a handler
- A two-value `Literal` keeps the triage decision inside the graph's edges

### 3. Measuring the router
- Measure against both baselines — "the router was cheap" means nothing without all-cheap and all-reasoning beside it
- Triage is a model call on every request, so a router that escalates everything is worse than no router

### 4. The router's own errors
- Over-escalation costs money; under-escalation costs correctness — bias the router toward over-escalating
- Escalate on uncertainty before answering; do not demote after the fact (a reliable wrong-answer detector would make the router unnecessary)

### Next Steps
- **`03_routing_lcel_runnable_branch.ipynb`**: rebuild the content router from `01_routing_langgraph.ipynb` in LCEL with `RunnableBranch`

### Where this came from
- *Reasoning vs Non-Reasoning* and *Reasoning Effort Levers* notebooks in [repository 01 — foundations](https://github.com/Sourav692/ai-engineering-demystified-01-foundations)